In [2]:
import json
import random
import uuid

def generer_hex_id(length=32):
    """Génère un TraceID ou SpanID hexadécimal réaliste type OpenTelemetry."""
    return uuid.uuid4().hex[:length]

def generer_80_scenarios_reels():
    scenarios = []
    
    ips_externes = [
        "185.220.101.5", "178.62.204.99", "89.187.160.14", "45.83.64.12",
        "103.149.1.208", "154.12.1.136", "188.2.1.194", "194.26.29.11"
    ]
    
    payloads_injection = [
        "' OR '1'='1",
        "' UNION SELECT * FROM users--",
        "'; DROP TABLE orders;--",
        "1' OR '1'='1' /*",
        "admin' --"
    ]
    
    base_ts = 1782900000.0
    
    # -------------------------------------------------------------------------
    # 1. 20 CAS - DDoS Volumétrique (frontend)
    # Signature réelle : Saturation descripteurs de fichiers, CPU contraint au cgroup,
    # drops Envoy/gRPC "ResourceExhausted" ou "socket: too many open files".
    # -------------------------------------------------------------------------
    for i in range(1, 21):
        cpu = round(random.uniform(98.8, 100.0), 1)
        rx_bytes = round(random.uniform(145000000.0, 210000000.0), 1) # ~150-200 MB/s
        ts = round(base_ts + (i * 315.4), 1)
        ip = random.choice(ips_externes)
        trace_id = generer_hex_id(32)
        span_id = generer_hex_id(16)
        duree = random.randint(10200, 15400)
        
        scenarios.append({
            "nom_cas": f"Cas {len(scenarios)+1} - frontend (Attaque (DDoS Volumétrique))",
            "input": (
                f"=== MÉTRIQUES (cAdvisor / Prometheus) ===\n"
                f"container_name: frontend | cpu_usage_seconds_total: {cpu}% | "
                f"network_receive_bytes_total: {rx_bytes} | tcp_curr_estab: 14250 | timestamp: {ts}\n"
                f"=== LOGS APPLICATIFS (Kubelet / CRI) ===\n"
                f'{{"ts":{ts},"level":"error","caller":"main.go:142","msg":"error while proxying request","error":"rpc error: code = ResourceExhausted desc = grpc: out of memory / socket: too many open files"}}\n'
                f'{{"ts":{ts+0.1},"level":"warn","caller":"middleware.go:84","msg":"upstream connection timed out","client_ip":"{ip}","status":503}}\n'
                f"=== TRACES (OpenTelemetry / Jaeger) ===\n"
                f'{{"traceID":"{trace_id}","spanID":"{span_id}","serviceName":"frontend","operationName":"ingress.HTTP/GET","durationMs":{duree},"http.status_code":503,"error":true}}'
            ),
            "reponse_attendue": {
                "service_defaillant": "frontend",
                "type_panne": "Attaque (DDoS Volumétrique)",
                "raison": "L'augmentation massive du trafic entrant (rx_bytes) et la saturation du cgroup CPU ont provoqué un épuisement des sockets TCP et des erreurs gRPC ResourceExhausted (HTTP 503)."
            }
        })

    # -------------------------------------------------------------------------
    # 2. 20 CAS - Brute Force / Credential Stuffing (checkoutservice)
    # Signature réelle : CPU/Mémoire normaux, rafale de codes HTTP 401 / gRPC 16 (Unauthenticated)
    # provenant d'une même adresse IP avec latence très faible.
    # -------------------------------------------------------------------------
    for i in range(1, 21):
        cpu = round(random.uniform(11.2, 18.5), 1)
        rx_bytes = round(random.uniform(12000.0, 35000.0), 1)
        ts = round(base_ts + (i * 240.2), 1)
        ip = random.choice(ips_externes)
        user_cible = random.choice(["admin", "root", "test_user", "administrator", "service_account"])
        trace_id = generer_hex_id(32)
        span_id = generer_hex_id(16)
        duree = random.randint(12, 38)
        
        scenarios.append({
            "nom_cas": f"Cas {len(scenarios)+1} - checkoutservice (Attaque (Brute Force))",
            "input": (
                f"=== MÉTRIQUES (cAdvisor / Prometheus) ===\n"
                f"container_name: checkoutservice | cpu_usage_seconds_total: {cpu}% | "
                f"network_receive_bytes_total: {rx_bytes} | tcp_curr_estab: 42 | timestamp: {ts}\n"
                f"=== LOGS APPLICATIFS (Kubelet / CRI) ===\n"
                f'{{"ts":{ts},"level":"warn","caller":"handler.go:88","msg":"user authentication failed","user_id":"{user_cible}","client_ip":"{ip}","grpc.code":"Unauthenticated"}}\n'
                f'{{"ts":{ts+0.05},"level":"warn","caller":"handler.go:88","msg":"user authentication failed","user_id":"{user_cible}","client_ip":"{ip}","grpc.code":"Unauthenticated"}}\n'
                f"=== TRACES (OpenTelemetry / Jaeger) ===\n"
                f'{{"traceID":"{trace_id}","spanID":"{span_id}","serviceName":"checkoutservice","operationName":"hipstershop.CheckoutService/PlaceOrder","durationMs":{duree},"grpc.status_code":16,"error":true}}'
            ),
            "reponse_attendue": {
                "service_defaillant": "checkoutservice",
                "type_panne": "Attaque (Brute Force)",
                "raison": "L'apparition répétée d'échecs d'authentification (gRPC Unauthenticated / Code 16) à faible latence depuis la même adresse IP indique une tentative d'énumération d'identifiants."
            }
        })

    # -------------------------------------------------------------------------
    # 3. 20 CAS - SQL / NoSQL Injection (cartservice)
    # Signature réelle : cartservice en C# / ASP.NET et Redis.
    # Erreur de syntaxe du driver StackExchange.Redis et exception 500 dans la trace.
    # -------------------------------------------------------------------------
    for i in range(1, 21):
        cpu = round(random.uniform(38.0, 52.0), 1)
        mem = random.randint(195, 235)
        ts = round(base_ts + (i * 180.7), 1)
        payload = random.choice(payloads_injection)
        trace_id = generer_hex_id(32)
        span_id = generer_hex_id(16)
        duree = random.randint(1100, 2400)
        
        scenarios.append({
            "nom_cas": f"Cas {len(scenarios)+1} - cartservice (Attaque (SQL Injection))",
            "input": (
                f"=== MÉTRIQUES (cAdvisor / Prometheus) ===\n"
                f"container_name: cartservice | cpu_usage_seconds_total: {cpu}% | "
                f"memory_working_set_bytes: {mem}MiB | timestamp: {ts}\n"
                f"=== LOGS APPLICATIFS (Kubelet / CRI) ===\n"
                f'[Error] StackExchange.Redis.RedisServerException: ERR syntax error at or near "{payload[:8]}"\n'
                f'   at StackExchange.Redis.ConnectionMultiplexer.ExecuteSyncImpl[T](Message message, ResultProcessor`1 processor, ServerEndPoint server) in /_/src/StackExchange.Redis/ConnectionMultiplexer.cs:line 2841\n'
                f'[Critical] Unhandled exception in ASP.NET Core request pipeline: payload="{payload}"\n'
                f"=== TRACES (OpenTelemetry / Jaeger) ===\n"
                f'{{"traceID":"{trace_id}","spanID":"{span_id}","serviceName":"cartservice","operationName":"hipstershop.CartService/AddItem","durationMs":{duree},"grpc.status_code":2,"error":true}}'
            ),
            "reponse_attendue": {
                "service_defaillant": "cartservice",
                "type_panne": "Attaque (SQL Injection)",
                "raison": "Une chaîne malformée contenant des caractères de contrôle SQL/NoSQL a généré une exception de syntaxe dans le driver de base de données (RedisServerException) et une erreur RPC de code 2 (Unknown)."
            }
        })

    # -------------------------------------------------------------------------
    # 4. 20 CAS - SSRF (checkoutservice)
    # Signature réelle : Requête sortante anormale vers l'adresse IP de métadonnées cloud
    # (169.254.169.254) provoquant un "context deadline exceeded" (gRPC code 4 / 504).
    # -------------------------------------------------------------------------
    for i in range(1, 21):
        cpu = round(random.uniform(18.0, 28.0), 1)
        tx_bytes = round(random.uniform(88000.0, 99000.0), 1)
        ts = round(base_ts + (i * 290.5), 1)
        url_metadata = random.choice([
            "http://169.254.169.254/latest/meta-data/iam/security-credentials/",
            "http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/",
            "http://169.254.169.254/latest/user-data"
        ])
        trace_id = generer_hex_id(32)
        span_id = generer_hex_id(16)
        duree = random.randint(5000, 5012)
        
        scenarios.append({
            "nom_cas": f"Cas {len(scenarios)+1} - checkoutservice (Attaque (SSRF))",
            "input": (
                f"=== MÉTRIQUES (cAdvisor / Prometheus) ===\n"
                f"container_name: checkoutservice | cpu_usage_seconds_total: {cpu}% | "
                f"network_transmit_bytes_total: {tx_bytes} | timestamp: {ts}\n"
                f"=== LOGS APPLICATIFS (Kubelet / CRI) ===\n"
                f'{{"ts":{ts},"level":"error","caller":"shipping.go:112","msg":"failed to fetch shipping rates from upstream","url":"{url_metadata}","error":"Get \\"{url_metadata}\\": context deadline exceeded (Client.Timeout exceeded while awaiting headers)"}}\n'
                f"=== TRACES (OpenTelemetry / Jaeger) ===\n"
                f'{{"traceID":"{trace_id}","spanID":"{span_id}","serviceName":"checkoutservice","operationName":"hipstershop.CheckoutService/PlaceOrder","durationMs":{duree},"grpc.status_code":4,"rpc.status_message":"DeadlineExceeded","error":true}}'
            ),
            "reponse_attendue": {
                "service_defaillant": "checkoutservice",
                "type_panne": "Attaque (SSRF)",
                "raison": "Une tentative de connexion non autorisée vers un point de terminaison d'infrastructure cloud interne (169.254.169.254) a entraîné une expiration de délai (DeadlineExceeded / gRPC code 4)."
            }
        })

    return scenarios

if __name__ == "__main__":
    data = generer_80_scenarios_reels()
    
    with open("benchmark_80_attaques_reelles.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
        
    print(f"✅ 'benchmark_80_attaques_reelles.json' généré avec succès ({len(data)} scénarios).")

✅ 'benchmark_80_attaques_reelles.json' généré avec succès (80 scénarios).
